# Validate the Input Raster Files for the Simulation

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import os

from _configs.country_config import *
from _configs.files_config import *
from _configs.run_config import *

from _utils.utils import *

In [2]:
if Path.cwd().name == "population_validation":
    os.chdir(Path.cwd().parent)

print(f"Current Directory: {Path.cwd()}")

Current Directory: /work/tuv89272/Calibration_Pipeline_v7_AO


## Import Rasters

In [3]:
districts_raster, metadata = read_raster(districts_raster_path)
district_mapping_csv = pd.read_csv(district_mapping_csv_path)
observed_population_raster, _ = read_raster(observed_population_raster_path)
treatment_seeking_raster, _ = read_raster(treatment_seeking_raster_path)
travel_time_raster, _ = read_raster(travel_time_raster_path)
pfpr_raster, _ = read_raster(pfpr_raster_path)

seasonality_file = pd.read_csv(seasonality_file_path)

nodata = metadata["NODATA_value"]

## Basic Validation

### Valid Flattened Rasters

In [4]:
districts_raster_flattened = districts_raster[districts_raster != nodata].flatten()
observed_population_raster_flattened = observed_population_raster[observed_population_raster != nodata].flatten()
treatment_seeking_raster_flattened = treatment_seeking_raster[treatment_seeking_raster != nodata].flatten()
travel_time_raster_flattened = travel_time_raster[travel_time_raster != nodata].flatten()
pfpr_raster_flattened = pfpr_raster[pfpr_raster != nodata].flatten()

In [5]:
for raster in [districts_raster_flattened, observed_population_raster_flattened, treatment_seeking_raster_flattened, travel_time_raster_flattened, pfpr_raster_flattened]:
    if len(raster) != len(districts_raster_flattened):
        raise ValueError(f"Length mismatch: Raster length {len(raster)} does not match mapping CSV length {len(district_mapping_csv)}")

print(f"All rasters have consistent lengths: {len(districts_raster_flattened):,.0f} pixels.")

All rasters have consistent lengths: 50,745 pixels.


### Districts

In [6]:
district_mapping_csv

,ID,region_name
0,1,Bengo
1,2,Bie
2,3,Cabinda
3,4,Cuando cubango
4,5,Cuanza Norte
5,6,Cuanza Sul
6,7,Cunene
7,8,Huambo
8,9,Hulia
9,10,Luanda


In [7]:
districts_raster_unique = np.unique(districts_raster_flattened)
districts_raster_unique

# Check if these values match exactly with the unit_id values in the district_mapping_csv, and print okay if they do and print not okay if they don't match
district_mapping_unit_ids = district_mapping_csv[ID_COLUMN].unique()

if np.array_equal(districts_raster_unique, district_mapping_unit_ids):
    ok(f"The unique values in the {districts_raster_unique.shape[0]} districts raster match the unit_id values in the district_mapping_csv.")
else:
    error(f"The unique values in the {districts_raster_unique.shape[0]} districts raster do not match the unit_id values in the district_mapping_csv.")
    raise ValueError("Error: The unique values in the districts raster do not match the unit_id values in the district_mapping_csv.")


✓ The unique values in the 18 districts raster match the unit_id values in the district_mapping_csv.


### Population

In [8]:
total_population_observed_year = observed_population_raster_flattened.sum()

print(f"Total Population in Observed Year ({observed_population_year}): {total_population_observed_year:,.0f}")

if total_population_observed_year <= TARGET_POPULATION_INITIAL_YEAR:
    error(f"Total population in observed year ({observed_population_year}) is less than or equal to the target population in initial year ({TARGET_POPULATION_INITIAL_YEAR:,.0f}).")
    raise ValueError("Error: Total population in observed year is less than or equal to the target population in initial year.")
elif total_population_observed_year > TARGET_POPULATION_CALIBRATION_YEAR:
    error(f"Total population in observed year ({observed_population_year}) is greater than the target population in calibration year ({TARGET_POPULATION_CALIBRATION_YEAR:,.0f}).")
    raise ValueError("Error: Total population in observed year is greater than the target population in calibration year.")
else:
    ok(f"Total population in observed year ({observed_population_year}) is within the expected range between the target population in initial year ({TARGET_POPULATION_INITIAL_YEAR:,.0f}) and the target population in calibration year ({TARGET_POPULATION_CALIBRATION_YEAR:,.0f}).")

Total Population in Observed Year (2020): 33,497,946
✓ Total population in observed year (2020) is within the expected range between the target population in initial year (25,279,115) and the target population in calibration year (37,885,849).


### Treatment Seeking

In [9]:
error_count = 0
warning_count = 0
for treatment_seeking_value in treatment_seeking_raster_flattened:
    if treatment_seeking_value < 0 or treatment_seeking_value > 1:
        # error(f"Treatment seeking value {treatment_seeking_value} is out of range [0, 1].")
        error_count += 1
    elif treatment_seeking_value < 0.1 or treatment_seeking_value > 0.9:
        # warn(f"Treatment seeking value {treatment_seeking_value} is unusually low or high.")
        warning_count += 1

if error_count > 0:
    error(f"{error_count} treatment seeking values are out of range [0, 1].")
    raise ValueError(f"Error: {error_count} treatment seeking values are out of range [0, 1].")
elif warning_count > 0:
    warn(f"Warning: {warning_count} treatment seeking values are unusually low or high.")

### Travel Time

In [10]:
error_count = 0
for travel_time_value in travel_time_raster_flattened:
    if travel_time_value < 0:
        # error(f"Travel time value {travel_time_value} is negative.")
        error_count += 1

if error_count > 0:
    error(f"{error_count} travel time values are negative.")
    raise ValueError(f"Error: {error_count} travel time values are negative.")
else:
    ok(f"All travel time values are non-negative.")

✓ All travel time values are non-negative.


### PfPR

In [11]:
error_count = 0
for pfpr_value in pfpr_raster_flattened:
    if pfpr_value < 0 or pfpr_value > 1:
        # error(f"PfPR value {pfpr_value} is out of range [0, 1].")
        error_count += 1

if error_count > 0:
    error(f"{error_count} PfPR values are out of range [0, 1].")
    raise ValueError(f"Error: {error_count} PfPR values are out of range [0, 1].")
else:
    ok(f"All PfPR values are within the expected range [0, 1].")

✓ All PfPR values are within the expected range [0, 1].


### Seasonality

## Advanced Validation